# Original FP32 half-split RoPE — research candidate

**Prepared, GPU untested.** Only Qwen2 RoPE application changes; no QK norm, FP16 early cast, attention backend, model, task or LoRA changes. The exact source bundle below is the queue artifact.

20 CPU Triton-interpreter fixtures passed on Torch 2.15 CPU / Triton 3.9, not target CUDA/Triton 3.6. No speed, peak-memory, useful-share, quality or admission result exists. Useful original-kernel share >=5% is a particularly important unresolved gate.

Run on an explicitly authorized **cloud** single-visible-Tesla-T4 environment only. Reuse existing packages/model/data. Configuring this notebook does not submit to the queue. Final Kaggle admission is separate; GCP needs its own measured baseline.


In [ ]:
from pathlib import Path
import os

PREPARED_REPO = Path("/kaggle/working/soup-kernel-challenge")
PUBLIC_HUB_CACHE = Path(os.environ.get("HF_HUB_CACHE", "/root/.cache/huggingface/hub"))
RUN_ROOT = Path("/kaggle/working/rotary-paired-runs")
BACKEND = "KAGGLE_T4"
NATIVE_TIMEOUT_SECONDS = 900
RUN_GPU = False


## Verify and materialize the exact packet
This cell only writes a new temporary preparation directory and compiles Python syntax. It does not import the solution, access a GPU, download, install, or alter the prepared official repository.


In [ ]:
import base64, hashlib, json, tempfile, zipfile, io
PAYLOAD = {'candidate.zip': 'UEsDBBQAAAAIAAAAIVzahTyhzAAAAE0BAAAUAAAAc29sdXRpb24vX19pbml0X18ucHllT7tuwzAM3PUVhCZ7ydCxnTM3zQ8YhEylRGTSoSQU/vvKSjwUvY24B++895+SNlDjGwsmuOrlDLiuiQMWVoE7mVDKH6AxcuAmyQWtkB0MoBGIFggJeaH55L13M8Wesg3ju4OGaLrAybR5N+BlVSuwYgnf0+OH5G16Ml3Kscf9Z4+sHYacCa5VSvt5NlMb/IVFaIav3QKvV1mrBYJbRZvBKNbcFCxtQ0p9oB+d63UflSpNedE7/S0d1joFlWIYylHdqnSFUakm+zmM7hdQSwMEFAAAAAgAAAAhXP4inkdWAwAAeQcAABgAAABzb2x1dGlvbi9jcHVfY29udHJhY3QucHmVVW1v2zgM/p5fIfiTfdAZa3frhgH+EKzbsLstHXDphuFwEBSLcbTYkqOXNtmvH/XiLC2CAvMHW6LIhw8pki6K4t+B9z158/n2T636A9l58EDsoLfwmihN3txez4PQSLCUbMEo6HExaAE96TUXUnVEGyL0vQpbWxdFMZPDqI0j0oFxWvd2Ejht2s1sNhOwJgbWYEC1UO4QmJKWElu9nhF81kYPxBmu7FqbAYytoz9b7+5BXaYN+mVxSzI2H8f+wIx23BzYqC2DYRXR7jeyB9KB486Z8pwaJQXTRnZS8X46arkSUnAHBSXveG8hcwvPOQzSnBXXjN0bPADBWLQ34LxRZ3VPE5GTBHveupJTssruYwZrB9ZhAmpuLWaYtb22ELUo4U73zTNKTP7CzmNQiqtmaTxUEWTgdot8kXNDnkVJAsro0aRMays7tZJI4b9g9H9FyUP5KssnwlKN3tly1VyiZvOSks2uucD1ZhskAtevMMJVc4EbuJMtNEU7+iJH14ECgxEYZJf8vJ8kZVZPn6oeuAqBWQBR/vXy4vlFCi1WFl5daTd8PL2znPeEGjRUUqG/nDbH1ZFc9hZRdkgqQpeYZhciQ7WqqmOhjuEGMKjLqjaYP2nAss5wwcpkvH1svP0N48w9lUdCaVfUUQR4vJ+ay6vytJnqVGlTrzDrx7AAQYniTt4B4+K7lspFkxVg38HxDrB1mMF2s9hBUFY1lpuCTG1Ht7SlFnXzzSfxIYR70uBRKR11Qdn5sYdcYghmWS+3UO4r8ge5evHi+RVBAmSPmOSQrGA/Qot8j6S4dzrkqA6v8kBJ8ILp6LKbANBRCBA/5Fh2lk4QJ0WR2uthBsoucsXOqU5b41fKpnBo6wVnYWo2aUCcqit9zuRpld1vo9atVk52XntM/FOs2vOF9STZJ5mcTopULvR8tTyoYCEx3WOAEHEc4aRr9YgzAP8/7NPb5fx6vpyz+eKaLebLD1/esvn13zcfFkt2s/j4DQdxN3oGe2g90kyk6PEup8cZ6bR6rBYcGWgWWGw4I8UgrZVaNcXt4p/FzddFkdk3KQbG7vCvgwqMhY6Sa8JwhA7AWBiZBWMDl3hWpErKTfXdahX3owllFLa18MNoy9iNFEtRgHLNJebkJ1BLAwQUAAAACAAAACFcrJOKlZILAAA9IAAAGQAAAHNvbHV0aW9uL2dwdV9wcmVmbGlnaHQucHnNWVFz47YRftevQNkX0oFhS2c7N/JwMm5Od7lJxnZs3bQZ2+VAJCSxJkEaAGXpmv737gIkJcqW7SSdTHVzFgUuFovFt98uAM/zRssyS+PUkGllKiXI+IjMuBGUyMIQVUmSVCqVM1IqUXLFTVpIRq5znmVkmi5tl7xIRKZJIbMV6/XKlZkXkuznRBdZZeVnZRVB/2mWzuaG7O8XlSEHUjwetI0HJTfz3hko1UWlYnGwEEpDVw12VLlQacwzMikqmXC1Ilwm5HL8DxLPRXyvyZSnGYmzQouEkfFcgEmpBvt7Oi6UOOBJnmpURvAVIz8KJUVGxFLE1jwU5mRtn3YTFWSqipzERZYQo3gqwQus53leL83LQhnC1QwEtWh+z7meZ+mk+fkvXcjmudA9qwxnCSKkbr7ESdfPqtUDg8ViwuP7Xq+XiCnJYWw/GPYIfMqwGZWdqRl4RppL/KX84LRkPEkiXjf7nvWzR5V4qFIlknCsKhGc8rBkVgFKaj+wakEwRGN8zuAxYErA2i0EKIWfLL9PUuVDJ1CrrRoqlqk2UXEffuSZFk4JdKoyEyZpbHxtuKl06H08+/zT6INHEQHO4WCI7UPt4oTnhRS0XaHQ+3L+4/nF38896tY2vLlzyo1aOQ/gJ52CR5mQi1QhugRMdnz1eXxxHn0+H4+uLq9GYy9Yi1vjeKoFuaqkSXMxUqpQvvfp8svGssNDpYUmqTRCQTP8tcj2gvW49QIVKp5To1JTSPjiUk8LBSDVraBdbNbIq8IAardexuCRuJC42KYRTGVZGQ0rNgWgyhjdDG83p+0GZVFUx0cUkb+ExHvHTtihRwrlbHvyfsD6fXb4TVz1B++d2IbVT6SPWf+E9b3XHThC8whMDpaAjHHkg7E1EL7W+kkTy6SB4qZLp7XJcZVwlohFGgvwCwzjB38J++sp2fcwTlTLSJ4L/xBkvLHQGQfeeoPBF1KQRarTSQZEV3d7zqo1um6864svl9HVxfjs6pfo+4sPo0+j8+jD5yvvLtRG+RAfB15plhud/0ouVDpLJTAWsFbGcTCIh5WFFjoMTcoIMigHb81zYdKYGKGNBg5ZCGRdtkYLeGBCDY2BNSTx/T6FfwH1B7T/LR3AwxE97rsmfBrQo2AL+g/0nsZUhw5d/iQEdSEqDOMJdd4MPXTvxhzw8wi21c7nlSliriHOrBxNzKoUoXs3zQpu+idbg+JHLEsRY8C3gPZrW4InsuCXimehCxWGkQhUVpbZKnJNUVnoSOST5zXMdGiqMhO+MylLpS6BRP39k+PjdyfU/V0yTCWZH2xNGtluzkvhL5n9DqzPl+huZ1V3LDEL106ZKZ4w/OM3k6VoYUBnWxby53u5AXb0QTM4p0KgKV/T0vf3avk9PoP13mvH3BMzWHXLFr7t8VTRii4bNZuDBkOuIX0YsmIA5zQBzg/DZfvcjShL8Deeo2bvjsECCZn4lvExHkNPA+Jh5ZpM7dEWbYWYTqHOgBQSTbiJ5wC+YDNmRss5r7RJAf86nUmR7H8VqtiIkFMbTVBliH2LZHLv8nhbFWD9QTc0In6x0piogicC0zcxcwGMXyRVbBN/kgJMwBZg+olYFVBVfACCPPihPxgO2Dbrp5AQTFFka5Zf1EtqhNTALhmkRL+VYvU4/s0ho/vwvw9ffXYH7F4KbsKTYBuHrd6HWi86QGPEZ0L6iwACf/AEu5bIITaEj29ZzWU6QnRFG+t3Hz4AuwL9zP2AQa0kYXF3SyNdLG6GdDi4a6OjNQOGwcSVzqqiggKCouBgePS6ZKt+ZpUfDdsuEdf+w/q9eIkxaqZw8V7zRfRRFV+FvCouR8yyhr+0ndaR7MC+HmFHENeR6M8oBNTpW4LWiXbo+uWYfSlY3xBhoo2TyMVJhDCBeqnIJ5BzENc6dP7vxNfHVGmzX6gEwC4LOYV6FkrcOMPw1we6mkjI1nwzJQ1Bjpzzc6haV8DxCQqnOdsF/wbnNh/4XiqnXtD82O/8klzCj77YPzqEkLBfGBg7o2HBXMz474MNjL2HeHj/agCc/kbg7xzsfRfLpzqMIfXBAu/3d6D8ORSf/inoXc+Y3rePb8Xzjs5/HsJbdEY1OqMWnRHs/Dqw30ohQsVY9FUynnM5Ewm5/nB5Rprq5eDTz2euqvl4+W5AHg7um4obkgi05lzfQ7EOexS9UX9htd4pl3GDMHM7Yc10UvKIG4gBt5V0uaLbGkHPR66SLZ1QQummwzV8Z+Ic5m8Ll46z0SyYCsLA7ZyI3cm9UuQNIO1ChfhCfdfEL8wtkX5dUFJIgL87yeCneJRChVvz8aHyiu7FKlrwrBLQr6hKHZ7QVEcxsBkEhJ1SRxFOu7YQglZbA9HE/rdb9nWq0QnkXjQ4hToPNxe18wT4jeBeszME7q7bVfKn8pka9g8XwXZxFL1X4VTuLn9daJhKyR3Y8a1fqdVEFxSnRRNVlLhzB9LFugsKnBBWb28PWPg4uDm86xZvsgn+dicFTRFu1MWaGbuVLq3Zau2jltECutH4WtG+pdbxguMMPJFwBahrYM6rXbd2K2kn7gQAiTzzYSJ099y2tgobmIqy9F74Itg7OXpTiW8pkkJhM3sLlzrBlyt6PqPAlcMmkUJelzPmZhjZEy3HnxRqyiwciP1jcHvzZOcO+2D5TPC8gWZbkowc4mp0R7MHwLYLm9B9dUj2S8utSkC58POjkIOzlv8s3zYVOZ4h4gGLPT2DDvackGAp8RK9utNE9oCKG3q0beCayLVyjS1v1IHJeQrJ2TF21NFrrf/evm+1ubXIuUTvaiES/+jb/rvBRo0wnYUbHf15msD2AtLSVxH2caOJZLcO4LngiYbApF0OdM0D+iwVwMceQ+UiSQHDTvc7p6IeLuMrmGnYX8dhtMEHu7SChIjweDPH8y0d/tuzLRhz3tADPuQAG4/WrXPI/t6wHx0eHuJ/9p+OExhOUkaW6fHE0foXtoCAJq+Vg5WoMhHCF+tCxQcN9DCwEevbFIOnnOsSJ5+sO11ZNhnlE5EkgALs2nR8OVM+kyid+55Jfbh025mlk+meCTO7qw3xWMKprVkI/AsR4j9Jv6yS+qES4iueXP3p2W4BpT0eRumwsXbPrQ5bAwLY8hRSs4HYzjd95JjSdXtCl0V93GUX7Dn+/8OJKHSGNnZbt1uOwjR3upOu2xnTZk5dvZ1D5eZT55c6qdnzAUca9aT8p9nbnlc2Z1fR9xdfzsfXN16duSFSkWojl95F4t09HdKl2Rcn+XK+eW2mTyf2uplhaOf1zaCbwnCls2f8tmvxwwYe/8ssvlVDdN5Bjqp9QGD7pkSyz6FggZYV0ZAPwcd4FIR3SDZbgYA7DtIk5ngmNIUS53RLo+KPxJ1tEXQ7HmNp2CrDBrrI8WIkcfdlSbpIE5AxBfKMIJzkqc4RQuz/tg7ANF4D3BYCdgO0uwLYWCG9AsCoQkJygg1xM9TmVQ+ACG1qO5dmOU2RgTRkX8isG8fnbJYVE9/bY/ZXWxU2PTZVtFdDHfLHuzV0Z9Oji1AjliZEEdi9wKYFf24Fcj2gvfYUTAsO8/SVdzuZ5tz/bnjLfr3VgUexZ0CtJvTf21RA/7zKfuVJEtyym3/eytO7van5+iZtXU2gxhqjZPDdLZu+G/wOHWDGb9HRerwDHvRx2HagEGmD45OwvgNl7qe/dvhkBaiGKGZzsUzSGSC8c6jgwMOqMsFgb24PL8+ur0cfosur0cefPn/6YRxdnP/0i0fBoKjGQGsbtVyl3e1jl+LqYcQyFqUhf+NajOwj1qtQSUL7cIcdAm+NQiVK5YNUQNub2bB9YlhycgOIjxuOesKQNcqbm0aGN8MA+EeFZxwWidjCkiovte8soKmEHGDCQfCNdyvrjXupoCh8Sba+h7UXX9crbUQ+WqbGP8TSoAlP51tk9x3uddVDP+j1oFdk79miCKSjCC+io8gbugvp3n8BUEsDBBQAAAAIAAAAIVxceNqY/QwAAJwkAAASAAAAc29sdXRpb24vcm90YXJ5LnB53Vptc9u4Ef6uX4HyplPKoWjJebmc79TpxXaczLW2z1GuuUldDkRCFGKKpAnQlvzr+yxAUpQo+V56X1pNIhOLxWLfsQvKcZzLQsYy5QnLSj3IZoM84aFgb6+eH7E5T2YDlSdSs+vs6uxblmb4Vyx4Ih+5llnKsoJxrUVqBjnX4dzv9a6KLCpDA4qkMlAmFTv5ePr9IEuTFeNpxGZyKaKB5uqWxSJbCF2sfHZy9ZHJVIsiL4S2O0jV4ykTS3ARSo3FhbgrhdIiYhp/aNO5x1JxL8AKy0UxIwZTiADezkHvHtxGllaYcLnwe47j9OQizwrNZmUa6ixLVA2YczVP5LQeylTlItT18IvK0vo5U71ZkS0MA1jBKvAVhnYizJJEGDWoevIkK0m6Xk1DZ0U4bwaF1GvyduQnPI1LHgvGFdNJr3f26ersZHJ2Gvz4z7OL4M3fL9+wMXOmL2Yi+ub1jM+4eC1eCPH8m6l4Pn0dRl+/ejUdha9fTodffxNGTi84ufx4MfmARRUvbr/X60VixgKoMNXurUwjjx0ciKUueP+4x/CpFn2muRv2bMxGBpyCysaUgcoZTQCHDJCyP7Ojl69oPLS06JMXMLLrXF9Ovr/+OTj76f3p2cXJGXPYM6NhPyoXuXIjGWrXMDW2TIXE8Thdc+cxBVUFt2KlxpOiFADMklLN7aCWS/F7EYRZJGKRumG2yGUiQC3lC1HJ9xW7zPVAwoMLMUtkPNfWT8W9jARcyWf/EFyVBXyuKGFN621KwEpzOLZI72WRpQtwysCLb2hGsoDxs2IFJWXKr3D8WEDwD5cfr4JK+pPL07NzmPL0/bXTrxW4XkyhUvNMQZRmml1kqWgpUy+xRY3jc7WwmwDueMypiNaWwXJMrFfTp+BSCXYN5cqFOCuKrHCdiwxhmYqCU6BdTT4xfs9lwqeJYAgwE4SS9NForA4VuHtryyLDhmMTFG4j1Oa0v7jFjJvzAvqzdvQQ71LpILutLFnjRzKmmB/XYeqrOYd7uRAJGiYbu/2+PxdLi+iuV7qGk0PmktnhZ05A3mbRPh+PXt0QzCedgcADYk8EGk5GlMmR/laF4xepK6eScQrB82D2/MhdNn50tuShZmXKYbpUxCbteFBNmJSRTGM2GPpDHwYkzxGwRjogRSJFke8gyRVCKcpUw8HSuhHyYFmkzF36OnN14peInOdHHptKHXKlrX7Yv9lw+XpoP/0Kc5ZkvIu6Q5Yi02A34HmerIw4nzx24rEPHvvZY++OkXb8EDlME3sem2wBNjyp+ZxuLzt5sw25+FWEkOFOftjEbJT9NjFnj1El4hQ+/MabeO889/TwqM/y+UrJkM61IkK0ZjODlxfZF+ulTMEXkVmtniWcCptgOi74IpCRO+yzA7s9XANTvEAqFu7Qs0DrWnRCYuUpOzxkRwbyBUOJrEczFgUAVxICgfqYemfguoG7hsoBe9en2YmZnRoy7Un8n9hNlzktdacEId76ZvoZm9PDKSVRgxbWaIj7kzfsT5SSRQItGcmalesFC7Pnd+zCjLjVCLwocj8BZ5l7bOGxDFosxkPLyWMHh9gAw13UcNjCPQFauINeOOog7SWo2gQ/7CGoRh2kvQS/QiKnBBfMENUlheH4LSd9cc0SjkJhDj98yBAvZRqZ1GfKHEV+lTZQHkXWo1YVfwBQOC7KxOU4w4Y4pKrhZhJ5pONs2Le8rEadxY9YPFovBi01qrABImcW7s+VnVbw0sXOuUb61cig7E8H0ZcMucawdl4nhNNPvzsj/IE5gbHzzsrzDlvnv5av8w5j+5PO/2GaiAmNkM7f0CQ9EJamB1r3hR5O/4iMErcj9hxI8Y6IjUcdpMpnDR//m4nlAoXAvWAKPYwYTHl4+8ALJBAeKfYg9Zw9G9okwsOwRHCj6FIm1aB9MlFoVyq/Ivdes5CjeKDjTT4Cl7NHUWQD1BEl1aiIYCrb4LyI7WlpTjudsTxT0vBByJbWYyUrZZR4ZFKKhZN4m/mpxhq20lS9/AHSCpdGqPN9YJivx2GL2BprtIVVbRktuwmTNjMZsyYVLbuJkfg2mbGm1KS80/rcAu1uQjzdOrFAu0qJpsIzFUWwrheaKg/Ov/TTSC4oAF6sS+mqVDNnhg1Oj6E11aAMnpdUq+aiXdRN2V/Z0FT48+ZJN09R6wlNlFGaGYISzEqlLsFcfTA/QDuDf+bvaN33lDl1kkGB3gXs33nslnooBftJFKVoZNBIi0cRQJQxRD/AbBnxgFqfqgyn9nw9ruRfKsoCm+SazoWnK1evcgFt1e2KaXL9CdSJxoGahyUKYlDpd1TnaIMEzqdo1ZVyaqobvJq8013L0TI4bTaWfkScELploaqIqS/FHJq70Eze1c8GLlWAwh26rUeIoi8YPs242SmwdAJIkPDHVXAvxUPDUaNaY8N6T7/m0KF5p0sYCgzM1KZkCV9lpV6LZj0i+gUu7aqGFBmn6+Z3RvTdc7c7iK6nA8uF0zj/XeP9dy3vx863dvh5eEMiTGnDGnR0bGAuLdyx291tUF8XNXI0XmpUS5zjiPoOsVlFFB5fjo5soN2Zqwk7qLcc3RDwqIo1M/96x9ZQnYJ4gbm5CsymHVYQD01qeE5iITbsLgShWTvATDMgLRDTsJqLMJz2N2dHTyskzMRsJkOJBnI3O1WrrgJULFHN0gZwRxwWXNpauEV+w2+Isg0OLeMyK5XbuA3R35p6mu2221RNGVzm89IvUwqPKKiAbt+PuOZBrovNgGwunipE0ijMWI9gX7qnROAowq/B5GlEZAN4fHSzg1tYogAf6O3Rq1eubvCddj6ne5mt5Csi94AXsaI7Kzr04zosqyXbSXobmXJoi2xq6oi6NnfjVgK2ZB3HOZWzmaC7FGlva3iSUMVhRJ3LGGfwwHbE1UWq+ra5Xa3o22qDwTXyuU+XpebkhUniyiUH0GhTyYbYn1nfbtK0S8cx+UEbYuW2tYl1RlBEUadsu+mZYrY9/9n3fY8dRzfeDnB0fFNVQxMUSpuVlYJsEHvFooz6MSq0InNfTKcL5KVrFlFAUBRdsjDlEKNapyqwHk3ra/JqyLXr2kfCUkEib4VLN48D+qKTc1CXTNPNVUiAnYXT/taiyg1IDSEKERdbPwMlOsPNCciCt0X2KFK6fq/44KWm5iPy39LdNXyxMv3fFN2Yh0gA8yyyJoPHwOqkETfUS48tO/5iDKhRUQgdLKDhQpqK0uhZuaaU6W8i0mUqiAa1rt3NEqDBI0fx7KMJb1JOmScoDOwcFLEuZJqlq0aHYpHrVZUaIrchuF4DRVJ2GFfHPIbmRB3XR/ua6N46jD6pgaXlQiRIKo1XGznqe9dx547ss1t1zWEk711UU0cvX/W9/o27bJdYq2ZjbyPdeyzd243u/Zh2cox9sLpcBNB9rsYvvD2XFmvp991/O9syta5tq3cBTuU9gb3+CKy3isiBfCTKOJFKtwxqDQID1bUXZQFjrgazMl6/v51jV70nvHjtbOTGcct5kfTjOklu3Wqvk7LX+m5Z15qJclfl2DjVbYpok7fuKO1RGVhtR+0jra0wm2ADk2CDzWzdUm+LvScy+k6uv2JXmXlDxqjzVJThTO2wQGiZFxBXqwkx/BfFqAatLqHvxSGVsSWeBd0HV4mOPrHJ60iIWQI+TOnbjGzpu759X+6Jzp0BX7tDvBWf8dPx2dDaiNApmhx9EB0ePhmd6yurvfEZt+MzWv6RAcoO4lZy2hOuv53qfx3fa63siPA6sp4O8cYoEKzMIaTgi6Ad2fETkU1abnmyrWQgCxVUJvdUjOY4JcVi+kuNat2F16+J6bVK9xXb5Pr95PIieH8xObu+uj6bOKZZdkZ1wUbVlrni+A1N8i8Yb90822p+zV+7kd6eazpny0Gr9jRv5X6iK53qndzHtKkq2cyUBfY1vdUfm9INNB6OzYtUS66+gPogck7v8lhdNKh1ESTTSOQCXym9XreED8uUDMR+PPyBTQsOz6hvn+rCdV2X+MaKpLpW6urOb9wWWCcwPw0I7h5EelQ5QZ1XzQt0dCGpotf5olD+Ag6eKN8g16/TDQw6tCToFTkgZWJTR1b/rmFcQf1d3lZrP6bfG+DgqlchfoL6uV4SokGknxIIBIcNwk6nQNckVk38gbwSenWrF6PklzOEpmu56VOETh1Ktjyqkuw0yaabLzdH7tQxUDIpHawJCIJ2v9+878TE1PnX0NgcE523nxDOUEAP2f3xwHEnIVibFMIGqMrKIhTBQqoFwSF4NjVeE42JaCfW7eWXPcmb31b4D+gkVKPafnOsxyUyD86PXxfxvz9wN16BVyR2vktvK6I6mev+KdgOqi3Ja+l+QZg9fFVEfm9W7FuVV/r093suNNe46BNRQTWBpdXr+gb8WUMp9oyw/kG+YBwC/vGA0yO4FThEEjX+3K0yd59MNxv9kOHxP1BLAwQUAAAACAAAACFcF7YI6Y4KAAA4IAAAFwAAAHNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5tVlZj+O4EX7vX6FVHlby0IolH+PxQEAWkw6wSDCz6fQ8GYZAS7SstUTJpNRtT5D/nuKhy7b6CDbeHVukqsg6vjrINk3zG0vihOLU2CWnsmKEfzYSWhJWMALfBkxUackNzIhB89L48v2vvxhhHpFxTChhuExyapCnJCI0JI5pmndJVuSsNBLgLvM85fVE3jwV55Lwsh6VOQv3dzuWZ4YTFlUQ5rRkOIQV9EK0qEqOQJIdYWIXZJATvNcsNRnLS8zOd3e/fn28f/jt4f7R8GFLh9CnhOXUiUlpmY8Pvz5++xo0JKZt+L5huubd3+8fvt7/A1iUbE6G2cHhh6RIdpZQu2ERcmCeU9/88tv3nqXIqUiTMCnTsxElHG9TEpn23d3dX7pLFpjhjJQs+UEsM9yayFi7yNvYL1DxEKdEEjpoMZ9PF44gVwLfRWRnCL5gl1CweMBLHJMowCwp97BCElrhFskV7NWdAZ8jOqAQcdBU2RXe++HWlu+eCOPgTvFyfXICPTR2OTNOQG5YmtneSHJOccH3eanpI1LicG/ZTpjmlFj2IB+jMXBItwu3BDAWcpek5ZV0YFASliQC4sb3zVKSAlBQAXJ97XxnV3GhfFGk50BNBUXOA5Jt+3yxELmsipRYSow0obzAsPxY2kpZDJ0cWmUktWzbgTDY44JYJ0f+dpSrpdQyt6rhqsxjhiNHfFk1GRKS2CiuNRiiV6pdUItNMQLBxMY/kqKm6i+uPa1MCBQWsNjNFOYcAtPADgd8RcJPEAKnZnRNF+ESB0XJgPInQdmOuzI1EsWIxPbqcmdpLqQRhQRyaobGM/VbjhpgdTTR4nRgCWLrxwt1T3IBbV/FpixMjmAsC+CGboJPBquOq+FwTPE5r0oRj6bIVElc5RU3kQk+wDSCmEcmhzwgHyiJIUE+keApIc8mhG0TrlUBBic4C9RylvoZitHSnyp1BL+AR0Io6NkxT7Iz1BIyoTWyrCByIKlTbQCILW5ZLpL/AaoVXYA5rDWwVEfHW4vV8dBwt3hWm400xXo1djfIqEcwGHm2vXYcB61W3qbh720RgAGl7azYvpCrb1qDpJwY8WVwd0zVz0a2jtaB4LtON5dhO8T45jx0K7IhWto4gshpoki8eBs62RGxAxLbC4Raj6wiSH5NYMd25MLobxhs1g7lg5qb2F2oJoCkgsAXBdhSqRk4QeKy3W0IuVvfRaXvKiWPkEaPVQK5NBDWCoDf/ny4mjzoagRFTFTR1p2nCycKSJz67P3KgdoCce3Rd9YMnUnWFxu2Iqm9NgKda2mYzYsYq2Vci/03Dq8yy0at0sjAaZo/a4v7wjsvQ09t/97VbuOumwvEK258hWheNTUB11Nt5gXcDMK1gRI57XHFZdjyJKZg9x+E5QFVkSzTZoCj33NoqywtxRNOKwUBqXNJKM+ZlSa8tJoO0ylYHlWw83rioDH8g05p7DobxAC4uPQXto53KBpGSqilFtWgbNYWssj0SCE9QnJyoA2FtiDnxBLjS5wq9gOwH696n9u0KjLU5usVgrTX9BViU9iizbbKfZrSW81eJL2rM2/DMVs1DCK9H3VcIPyHxMJbstXL3ZBOf1aMYvu1Tuia9nr/ftfRkUBOaPU0zmqExUp3Eq8nm5twpTnVbXWYCujzQFRLXm1pzjLIlB2MXiG0huIuzXFp/ZzQ3c92PRj3RhRTGLhkPJsAauWPK1r8GpxidUch2VraHRgsAabL/wNKX95zeQN6rfa7FLwRIlHfb9D9UfiTJd5qFUKH5rEGHhrEE2lLL3oJb4Ol2RphNMIxIHJE0AhwN1yoGyiFOdnt4HQoimhWlfLYHDDyuywBwfYc1DJYwz2g2xaAd9ordHAE/tULPMPZsD7nMpwArq2HCjyVkXvGcoYyXIZ738zyKNklcPzang1MhRgpnJCMvNCnfrPbnUMIqaLjbHF4eMYMFLllh30S7wkLchbBt47HHdQmwfW66v3mMkiTg8BQD8UXlfLthgINmnp9jdD6LSSawTR1RJBRoKmHfAEzxb4jChlkIy+yKVTJ5Ka2x4PbQ3ZEl7bBdcIkZHD/W3yk4evDunEj9JtYngMVhMWpTeXGLAd27cfhKx1W0W5PBUPLXpuFGEbm5jYIlEDq8kWUcZxtI2yIUBwdnle1oyE7F2KLQBGKUA2rCAc5Tc++6m4Ffa+l06RH8IRwtehtxJVP09/0yS5PJkdbQ+QVPq0Qqig/VoT8IEGUZL73Kpuzx+nOes8m+lhneVBO4H8b8cuZNy0jOhRxZuP6oc/UAUMh8kWQRAScW4oEJ1xe11oABRyxrSynB3KWlF10yNoFCTaDg7wDCQfaSOf4TKhXI0XOJXBGV7OYi5kqVULn9dWlr2edWxEuSTv7OxzAW5bMUjzIvMVkonpx+yU4imRRE/ZT7lsvuWrDK/gqU0pdtUCaTL6QV2GvafrGBTXZC6sJd+ttm3Sqx/9bN/jeu5j2bjPDEkgEM5AyxKI1q4//MrXkbS+G0VY7at3rx04oKs8F8es2BVqvqdfeCjSVTH+s8dSZfFrOl9PZHH5mrrdAY8/xPn3yFh+9T97E8z5O5mjiLL35dDpfeO5yOl1O3ClMzRezxcybLZef4GnuzvS1J8ExYSAXHoXGB9hga4+4qs46O0r3WtiJ8grObJY9CpvHD0A+4nYztpUGfV/KDRyZlJSW0OQKHX/yO1tcv2+avk4XV6WpuifybOSNRtDIzi8rcHNaWiiyK/O6C80ijzpohSYbfzaBz+dmxt34YzmlCPVtDffXmwZY0ufyxN/eV3Qaj8O1PFrsoa5AfM69Gn+oa7y76VGcderVjXAgL/tTgp+IBTrL2wclnbx8Og+Q6UXaC7JaSxF0BJLyrap8BpFiW55LriIn4epQ0txucUHnQBNl3aSG80WHVPTlmJ5vd2eQ/lOdLCBDAr3oUjt/44AMUcHGf2gyfwO/OEjs4CCh+s6gt+Y/xeCLfC9XUlpnmEJ6CTiBXDX76DYo38V+h8HaJ1FEaMCTH8R359MFolXW0XxPcMR915PToHEgj9d62kOXOUN8pLEyEiWinVPrqlX1Vik+g3a+i9pdIpYXYFYfDou3VoS3JNB3fIL136acEdFmrkxwHa7S0kR6dg+1zVy5AcSU+Of8p1HcEYqBN7MiJRlsLW3pmzwqsKlLpCgFPvw40ka/1BJawI0m8qCZ0LrwZduW8kHWjftsS6II3CvoFZVSWicF8G5ERYBOkbD1cITKs5IvenPFX5/SYIGYWFPbaVooa2Jf3BX6Nc9IV7bWcvXZUO522fEqLnu0mCn1fMVeryZFQqLxEUH5mcT+7V6+vXBrLyPr/kDa670Nytu6kz8ZjxC+Y9HkQl5T5zT5h1ptYuHvz+JqHRxo6Msy8SeUmOTi5vj8Z5wmEI1xBec27nQabf92W938LUJ0ek2nzWIum2vx0E3RcmJtNn24uVGd+OWd/+2FBq2jREPmhWwmaoV6V4u1JZB7SKPxl2/fvz7+a23CpDjOQuRWFBiVrM0xBb8MFHwbKPgmUPpyvi6A7yuRP3jdsxoavNlVd2R1V8RLCFVH7RlAR8qJ6tUQK/PU98h4DjmqfpLdGvT0VIfpfwFQSwECFAMUAAAACAAAACFc2oU8ocwAAABNAQAAFAAAAAAAAAAAAAAApIEAAAAAc29sdXRpb24vX19pbml0X18ucHlQSwECFAMUAAAACAAAACFc/iKeR1YDAAB5BwAAGAAAAAAAAAAAAAAApIH+AAAAc29sdXRpb24vY3B1X2NvbnRyYWN0LnB5UEsBAhQDFAAAAAgAAAAhXKyTipWSCwAAPSAAABkAAAAAAAAAAAAAAKSBigQAAHNvbHV0aW9uL2dwdV9wcmVmbGlnaHQucHlQSwECFAMUAAAACAAAACFcXHjamP0MAACcJAAAEgAAAAAAAAAAAAAApIFTEAAAc29sdXRpb24vcm90YXJ5LnB5UEsBAhQDFAAAAAgAAAAhXBe2COmOCgAAOCAAABcAAAAAAAAAAAAAAKSBgB0AAHNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5UEsFBgAAAAAFAAUAVAEAAEMoAAAAAA==', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICI0MGIyNjQ2MTRlYWY2YTA3MzhiOWYyNWYwZmY5MTUxYzJlMTAzMmU4NDA0M2EwZGRmYjYxOGZjNDA1YzRiOGExIiwKICAic29sdXRpb24vY3B1X2NvbnRyYWN0LnB5IjogIjYzNWM3Y2U3MTcxNjM0ZWUxOGZmNjEwMGU0OGY5MDE2Y2M3NjczNGM2YjMzZWI5MjZiNDVkYjM4MmZiYTE1NTAiLAogICJzb2x1dGlvbi9ncHVfcHJlZmxpZ2h0LnB5IjogIjNjYjYyZGVlNWI4OTdmZDY4OWQ5MWJmYTdmZGU1ZGUyNzllYjcyOTI3YmIxYTNhOWRlZjZkODJmODZmOTY3ZmQiLAogICJzb2x1dGlvbi9yb3RhcnkucHkiOiAiMTFhZjc4ZTZkNWVhOWVjYTY2MDRiMThkNTY1ZTg5OTAxNjdkMTQzNDJmYzNmMGY0YTIzYTkzZGNmZGRjYTRjNyIsCiAgInNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5IjogIjA1ZGZlMDlhOGEyZjJhOWJhYTg3MmQ3OTU4MzhjMWNlMmY1OGJhYWQ4YjJiMzE4N2FhYTQyZTY0NjlhMzgwNjIiCn0K', 'official_frozen_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'run_prepared.py': '...', 'check_kaggle_environment.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJGYWlsLWNsb3NlZCBwYWNrYWdlIGNvbXBhdGliaWxpdHkgY2hlY2s7IEdQVSBpbnZlbnRvcnkgb25seSB3aXRoIGV4cGxpY2l0IC0tZ3B1LiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixpbXBvcnRsaWIubWV0YWRhdGEsanNvbixvcyxwbGF0Zm9ybSxzdWJwcm9jZXNzLHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKRVhQRUNURUQ9eyd0b3JjaCc6JzIuMTEuMCtjdTEyOCcsJ3RyaXRvbic6JzMuNi4wJywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQpDT01NSVQ9J2M3YzJmY2QxNTk3MmNlOGI4OThkOGQzZmY3MzJlZTE5MmM4MTdjZWInCmRlZiBpbnNwZWN0KGdwdT1GYWxzZSxyZXBvPU5vbmUscmVxdWlyZV9mb3VyX2NwdXM9RmFsc2UpOgogZXJyb3JzPVtdO3ZlcnNpb25zPXt9CiBmb3IgbmFtZSx3YW50ZWQgaW4gRVhQRUNURUQuaXRlbXMoKToKICB0cnk6YWN0dWFsPWltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpCiAgZXhjZXB0IGltcG9ydGxpYi5tZXRhZGF0YS5QYWNrYWdlTm90Rm91bmRFcnJvcjphY3R1YWw9Tm9uZQogIHZlcnNpb25zW25hbWVdPWFjdHVhbAogIGlmIGFjdHVhbCE9d2FudGVkOmVycm9ycy5hcHBlbmQoZid7bmFtZX06IGV4cGVjdGVkIHt3YW50ZWR9LCBvYnNlcnZlZCB7YWN0dWFsfScpCiBpZiBzeXMudmVyc2lvbl9pbmZvWzozXSE9KDMsMTMsMTUpOmVycm9ycy5hcHBlbmQoJ0V4YWN0IFB5dGhvbjMuMTMuMTUgcmVxdWlyZWQnKQogYWZmaW5pdHk9c29ydGVkKG9zLnNjaGVkX2dldGFmZmluaXR5KDApKSBpZiBoYXNhdHRyKG9zLCdzY2hlZF9nZXRhZmZpbml0eScpIGVsc2UgTm9uZQogaWYgcmVxdWlyZV9mb3VyX2NwdXMgYW5kIChhZmZpbml0eSBpcyBOb25lIG9yIGxlbihhZmZpbml0eSkhPTQpOmVycm9ycy5hcHBlbmQoJ1Byb2Nlc3MgYWZmaW5pdHkgbXVzdCBiZSBleGFjdGx5IGZvdXIgbG9naWNhbCBDUFVzOyB0aGlzIGRvZXMgbm90IGVxdWF0ZSBDUFUgaGFyZHdhcmUnKQogb3V0PXsncHl0aG9uJzpwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpLCdleGVjdXRhYmxlJzpzeXMuZXhlY3V0YWJsZSwncGxhdGZvcm0nOnBsYXRmb3JtLnBsYXRmb3JtKCksJ3BhY2thZ2VzJzp2ZXJzaW9ucywnZXhwZWN0ZWRfcGFja2FnZXMnOkVYUEVDVEVELCdjcHVfYWZmaW5pdHknOmFmZmluaXR5LCdncHVfY2hlY2tlZCc6Z3B1LCdvZmZpY2lhbF9jb21taXRfZXhwZWN0ZWQnOkNPTU1JVCwnZXJyb3JzJzplcnJvcnMsJ2VxdWl2YWxlbnRfaGFyZHdhcmVfY2xhaW0nOkZhbHNlfQogaWYgcmVwbyBpcyBub3QgTm9uZToKICByb290PVBhdGgocmVwbykucmVzb2x2ZSgpO3I9c3VicHJvY2Vzcy5ydW4oWydnaXQnLCctQycsc3RyKHJvb3QpLCdyZXYtcGFyc2UnLCdIRUFEJ10sY2FwdHVyZV9vdXRwdXQ9VHJ1ZSx0ZXh0PVRydWUpCiAgb3V0WydyZXBvX2NvbW1pdCddPXIuc3Rkb3V0LnN0cmlwKCkKICBpZiByLnJldHVybmNvZGUgb3Igb3V0WydyZXBvX2NvbW1pdCddIT1DT01NSVQ6ZXJyb3JzLmFwcGVuZCgnT2ZmaWNpYWwgcmVwb3NpdG9yeSBjb21taXQgbWlzbWF0Y2gnKQogIHI9c3VicHJvY2Vzcy5ydW4oWydnaXQnLCctQycsc3RyKHJvb3QpLCdzdGF0dXMnLCctLXBvcmNlbGFpbicsJy0tdW50cmFja2VkLWZpbGVzPWFsbCcsJy0tJywnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlKTtvdXRbJ2Zyb3plbl9wYXRoc19zdGF0dXMnXT1yLnN0ZG91dAogIGlmIHIucmV0dXJuY29kZSBvciByLnN0ZG91dDplcnJvcnMuYXBwZW5kKCdPZmZpY2lhbCBoYXJuZXNzL3Rhc2svdGVzdHMgbXVzdCBiZSB1bmNoYW5nZWQsIGluY2x1ZGluZyB1bnRyYWNrZWQgZmlsZXMnKQogaWYgZ3B1OgogIHRyeToKICAgaW1wb3J0IHRvcmNoCiAgIG91dFsnY3VkYV92ZXJzaW9uJ109dG9yY2gudmVyc2lvbi5jdWRhO291dFsndG9yY2hfZ2l0X3ZlcnNpb24nXT10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uCiAgIGlmIHRvcmNoLnZlcnNpb24uY3VkYSE9JzEyLjgnOmVycm9ycy5hcHBlbmQoJ1RvcmNoIGNvbXBpbGVkIENVREExMi44IHJlcXVpcmVkJykKICAgb3V0Wyd2aXNpYmxlX2RldmljZV9jb3VudCddPXRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk7b3V0Wyd2aXNpYmxlX2RldmljZV9uYW1lcyddPVt0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZShpKSBmb3IgaSBpbiByYW5nZSh0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpKV0KICAgaWYgb3V0Wyd2aXNpYmxlX2RldmljZV9jb3VudCddIT0xIG9yIG91dFsndmlzaWJsZV9kZXZpY2VfbmFtZXMnXSE9WydUZXNsYSBUNCddOmVycm9ycy5hcHBlbmQoJ0V4YWN0bHkgb25lIHZpc2libGUgVGVzbGEgVDQgcmVxdWlyZWQnKQogICByPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PW5hbWUsZHJpdmVyX3ZlcnNpb24sbWVtb3J5LnRvdGFsJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyJ10sY2FwdHVyZV9vdXRwdXQ9VHJ1ZSx0ZXh0PVRydWUsdGltZW91dD0xMCk7b3V0WydudmlkaWFfc21pX2ludmVudG9yeSddPXIuc3Rkb3V0LnN0cmlwKCkKICAgaWYgci5yZXR1cm5jb2RlOmVycm9ycy5hcHBlbmQoJ252aWRpYS1zbWkgaW52ZW50b3J5IGZhaWxlZDsgbmF0aXZlIG1lbW9yeSBzYW1wbGluZyB1bmF2YWlsYWJsZScpCiAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOmVycm9ycy5hcHBlbmQoJ0dQVSBpbnZlbnRvcnkgZmFpbGVkOiAnK3R5cGUoZSkuX19uYW1lX18pCiBvdXRbJ3Bhc3NlZCddPW5vdCBlcnJvcnM7cmV0dXJuIG91dAppZiBfX25hbWVfXz09J19fbWFpbl9fJzoKIHA9YXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXyk7cC5hZGRfYXJndW1lbnQoJy0tb3V0JyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1ncHUnLGFjdGlvbj0nc3RvcmVfdHJ1ZScpO3AuYWRkX2FyZ3VtZW50KCctLXJlcG8nKTtwLmFkZF9hcmd1bWVudCgnLS1yZXF1aXJlLWZvdXItY3B1cycsYWN0aW9uPSdzdG9yZV90cnVlJyk7YT1wLnBhcnNlX2FyZ3MoKTtyZXN1bHQ9aW5zcGVjdChhLmdwdSxhLnJlcG8sYS5yZXF1aXJlX2ZvdXJfY3B1cyk7UGF0aChhLm91dCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCxpbmRlbnQ9MikrJ1xuJyk7cHJpbnQoanNvbi5kdW1wcyh7J3Bhc3NlZCc6cmVzdWx0WydwYXNzZWQnXSwnZXJyb3JzJzpyZXN1bHRbJ2Vycm9ycyddLCdyZXBvcnQnOmEub3V0fSkpO3JhaXNlIFN5c3RlbUV4aXQobm90IHJlc3VsdFsncGFzc2VkJ10pCg==', 'prepare_asset_manifest.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWFkL2hhc2ggb25seSB0aGUgc2VsZWN0ZWQgcHVibGljIG1vZGVsIGNhY2hlICsgcHVibGljIHRhc2sgZmlsZXM7IG5vIGRvd25sb2FkLiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApNT0RFTD0nbW9kZWxzLS1Rd2VuLS1Rd2VuMi41LTEuNUItSW5zdHJ1Y3QnClJFVklTSU9OPSc5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2JwpEQVRBPXsnZGF0YS90cmFpbi5qc29ubCc6JzkzZDAzNmQ5MTJiZDgxYzMzN2RjMjk4OTNlMTNhNzZmMzYxNjFkOTZmODk2ZWY0ZTAyNDkyZTJmMWEyMWEwZGQnLCdkYXRhL2V2YWxfcHVibGljLmpzb25sJzonYjdmODViNzhiYTMxODZkY2U0ZGVhYjdjZmI4YmY4ZGNmNzk5OGUxNTE3YzU3NDlhYzBmNTAwNTg4NmM2ZjcwZSd9CmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDQqMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGIpCiByZXR1cm4gaC5oZXhkaWdlc3QoKQpkZWYgYnVpbGQoY2FjaGUscmVwbyxyZXZpc2lvbj1SRVZJU0lPTik6CiBjYWNoZT1QYXRoKGNhY2hlKS5yZXNvbHZlKCk7cmVwbz1QYXRoKHJlcG8pLnJlc29sdmUoKQogaWYgbm90IHJlLmZ1bGxtYXRjaCgnWzAtOWEtZl17NDB9JyxyZXZpc2lvbik6cmFpc2UgVmFsdWVFcnJvcignUmV2aXNpb24gbXVzdCBiZTQwaGV4JykKIHJvb3Q9Y2FjaGUvTU9ERUwKIGlmIHJvb3QuaXNfc3ltbGluaygpIG9yIG5vdCByb290LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NlbGVjdGVkIG1vZGVsIGNhY2hlIG1pc3Npbmcgb3Igc3ltbGlua2VkJykKIGZvciBkaXJlY3RvcnkgaW4gKHJvb3QvJ3JlZnMnLHJvb3QvJ3NuYXBzaG90cycscm9vdC8nYmxvYnMnKToKICBpZiBkaXJlY3RvcnkuaXNfc3ltbGluaygpIG9yIG5vdCBkaXJlY3RvcnkuaXNfZGlyKCk6cmFpc2UgVmFsdWVFcnJvcignTW9kZWwgbWV0YWRhdGEvYmxvYiBkaXJlY3RvcmllcyBtdXN0IGJlIHJlYWwgbG9jYWwgZGlyZWN0b3JpZXMnKQogcmVmPXJvb3QvJ3JlZnMvbWFpbicKIGlmIHJlZi5pc19zeW1saW5rKCkgb3Igbm90IHJlZi5pc19maWxlKCkgb3IgcmVmLnJlYWRfdGV4dCgpLnN0cmlwKCkhPXJldmlzaW9uOnJhaXNlIFZhbHVlRXJyb3IoJ3JlZnMvbWFpbiBtdXN0IGlkZW50aWZ5IGV4YWN0IHBpbm5lZCByZXZpc2lvbjsgbm8gc2lsZW50IHJldmlzaW9uIHN1YnN0aXR1dGlvbicpCiBzbmFwc2hvdD1yb290LydzbmFwc2hvdHMnL3JldmlzaW9uCiBpZiBzbmFwc2hvdC5pc19zeW1saW5rKCkgb3Igbm90IHNuYXBzaG90LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NuYXBzaG90IGRpcmVjdG9yeSBhYnNlbnQ7IHJlc3RvcmUgdmFsaWRhdGVkIHJldGFpbmVkIGxpbmtzIHNlcGFyYXRlbHksIGRvIG5vdCBndWVzcycpCiBmaWxlcz17c3RyKHJlZi5yZWxhdGl2ZV90byhjYWNoZSkpOnNoYShyZWYpfTttZW1vPXt9CiBmb3IgcCBpbiBzb3J0ZWQoc25hcHNob3Qucmdsb2IoJyonKSk6CiAgaWYgcC5pc19kaXIoKToKICAgaWYgcC5pc19zeW1saW5rKCk6cmFpc2UgVmFsdWVFcnJvcignTGlua2VkIHNuYXBzaG90IGRpcmVjdG9yeSByZWplY3RlZCcpCiAgIGNvbnRpbnVlCiAgcmVzb2x2ZWQ9cC5yZXNvbHZlKHN0cmljdD1UcnVlKQogIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90bygocm9vdC8nYmxvYnMnKS5yZXNvbHZlKCkpIG9yIG5vdCByZXNvbHZlZC5pc19maWxlKCk6cmFpc2UgVmFsdWVFcnJvcignU25hcHNob3QgdGFyZ2V0IG11c3QgcmVtYWluIGluc2lkZSBzZWxlY3RlZCBtb2RlbCBibG9icycpCiAgaWYgcmVzb2x2ZWQgbm90IGluIG1lbW86bWVtb1tyZXNvbHZlZF09c2hhKHJlc29sdmVkKQogIGZpbGVzW3N0cihwLnJlbGF0aXZlX3RvKGNhY2hlKSldPW1lbW9bcmVzb2x2ZWRdCiBpZiBsZW4oZmlsZXMpPDI6cmFpc2UgVmFsdWVFcnJvcignRW1wdHkgbW9kZWwgc25hcHNob3QnKQogZGF0YT17fQogZm9yIG5hbWUsd2FudGVkIGluIERBVEEuaXRlbXMoKToKICBwPXJlcG8vbmFtZQogIGlmIG5vdCBwLmlzX2ZpbGUoKSBvciBwLmlzX3N5bWxpbmsoKSBvciBub3QgcC5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocmVwbykgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgVmFsdWVFcnJvcignUHVibGljIGRhdGEgaGFzaCBtaXNtYXRjaDogJytuYW1lKQogIGRhdGFbbmFtZV09d2FudGVkCiByZXR1cm4geydtb2RlbF9pZCc6J1F3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN0JywncmV2aXNpb24nOnJldmlzaW9uLCdmaWxlcyc6ZmlsZXN9LGRhdGEKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pO3AuYWRkX2FyZ3VtZW50KCctLWNhY2hlJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1yZXBvJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCk7bWFuaWZlc3QsZGF0YT1idWlsZChhLmNhY2hlLGEucmVwbyk7b3V0PVBhdGgoYS5vdXQpCiB3aXRoIG91dC5vcGVuKCd4JykgYXMgZjpqc29uLmR1bXAobWFuaWZlc3QsZixpbmRlbnQ9Mik7Zi53cml0ZSgnXG4nKQogcHJpbnQoanNvbi5kdW1wcyh7J21vZGVsX2NhY2hlX21hbmlmZXN0JzpzdHIob3V0KSwnbW9kZWxfY2FjaGVfbWFuaWZlc3Rfc2hhMjU2JzpzaGEob3V0KSwnZGF0YV9maWxlcyc6ZGF0YX0saW5kZW50PTIpKQo=', 'requirements-kaggle.txt': 'IyBFeGFjdCBkaXJlY3QgdmVyc2lvbnMgb2JzZXJ2ZWQgb24gdGhlIHByZXBhcmVkIEthZ2dsZSBzZXNzaW9uLiBObyBzaWxlbnQgc3Vic3RpdHV0aW9ucy4KIyBQeXRob24gTVVTVCBiZSBleGFjdGx5My4xMy4xNSAobm90IGluc3RhbGxlZCBieSBwaXApLiBUcmFuc2l0aXZlIHJlc29sdXRpb24gaXMgbm90IGEgZnVsbCBLYWdnbGUgZnJlZXplLgotLWV4dHJhLWluZGV4LXVybCBodHRwczovL2Rvd25sb2FkLnB5dG9yY2gub3JnL3dobC9jdTEyOAp0b3JjaD09Mi4xMS4wK2N1MTI4CnRyaXRvbj09My42LjAKc291cC1jbGlbdHJhaW5dPT0wLjc3LjAKdHJhbnNmb3JtZXJzPT01LjE2LjEKcGVmdD09MC4yMC4wCnRybD09MC4yOS4xCmJpdHNhbmRieXRlcz09MC41MC4yCmFjY2VsZXJhdGU9PTEuMTQuMApkYXRhc2V0cz09NC44LjUK', 'submission.template.json': 'ewogICJjYW5kaWRhdGVfaWQiOiAicm90YXJ5LWZwMzItbmF0aXZlLWFkam9pbnQtdjEiLAogICJiYWNrZW5kIjogIkdDUCIsCiAgImJhc2VsaW5lX2lkIjogIlJFUExBQ0VfV0lUSF9NRUFTVVJFRF9HQ1BfQkFTRUxJTkVfSUQiLAogICJlbnRyeXBvaW50IjogInNvdXBfc2NyZWVuX3YxIiwKICAiYXJ0aWZhY3Rfa2luZCI6ICJzb2x1dGlvbl9idW5kbGUiLAogICJwcmlvcml0eSI6IDUwLAogICJjb250YWluc19zZWNyZXRzIjogZmFsc2UKfQo=', 'README.md': 'IyBPcmlnaW5hbCBGUDMyIFJvUEUgYXBwbGljYXRpb24gY2FuZGlkYXRlIOKAlCBwcmVwYXJlZCwgR1BVIHVudGVzdGVkCgoqKkh5cG90aGVzaXM6KiogcmVwbGFjZSBRd2VuMidzIGhhbGYtc3BsaXQgRlAzMiByb3RhcnkgYXBwbGljYXRpb24gd2l0aCBvcmlnaW5hbCBvdXQtb2YtcGxhY2UgVHJpdG9uIGZvcndhcmQvYWRqb2ludCBrZXJuZWxzIHRvIHJlZHVjZSBwb2ludHdpc2UgbGF1bmNoZXMgYW5kIHRyYW5zaWVudCB0cmFmZmljLiBPbmx5IFJvUEUgYXBwbGljYXRpb24gY2hhbmdlcy4gTm8gUUsgbm9ybWFsaXphdGlvbiwgZWFybHkgRlAxNiBjYXN0LCBmcmVxdWVuY3kvdGFibGUgY2FjaGUsIGF0dGVudGlvbiBiYWNrZW5kLCBtb2RlbC90YXNrL0xvUkEsIG9wdGltaXplciwgZGF0YSBvcmRlciwgb3IgcHJlY2lzaW9uIHBvbGljeSBjaGFuZ2VzLgoKKipDdXJyZW50IHJlc3VsdDoqKiAyMCBvcmlnaW5hbCBDUFUgVHJpdG9uLWludGVycHJldGVyIGZpeHR1cmVzIHBhc3NlZCwgaW5jbHVkaW5nIGEgcmVhbCB0aW55IFF3ZW4yQXR0ZW50aW9uIHJvdXRpbmcgZml4dHVyZS4gVGhlIDc5IHVuY2hhbmdlZCBvZmZpY2lhbCBTb3VwIENQVSByZWdyZXNzaW9uIHRlc3RzIGFsc28gcGFzc2VkIHdpdGggdGhpcyBzb2x1dGlvbiBhcHBsaWVkIGluIGEgdGVtcG9yYXJ5IHRlc3QgdHJlZS4gVGhpcyBtYWNoaW5lIGhhcyBUb3JjaCAqKjIuMTUuMCtjcHUqKiBhbmQgVHJpdG9uICoqMy45LjAqKiwgbm90IHRhcmdldCBUb3JjaCAyLjExLjArY3UxMjggLyBUcml0b24gMy42LjAuIFRoZXJlIGhhcyBiZWVuICoqbm8gR1BVIGV4ZWN1dGlvbiwgY29tcGlsYXRpb24gdG8gU003NSwgd2hvbGUtdGFzayBydW4sIFQ0IHNwZWVkL3BlYWstbWVtb3J5IG1lYXN1cmVtZW50LCBxdWFsaXR5IGV2YWx1YXRpb24sIHVzZWZ1bC1zaGFyZSBtZWFzdXJlbWVudCwgcXVldWUgc3VibWlzc2lvbiwgb3Igb2ZmaWNpYWwgYWRtaXNzaW9uKiouIFNjb3JlIGlzIG51bGw7IGFkbWlzc2lvbiByZW1haW5zIFVOS05PV04uIEdQVSBwcmVmbGlnaHQgYW5kIHRyYWluaW5nIGFyZSBmdXR1cmUgb3B0LWluIG5vdGVib29rIHN0ZXBzLgoKIyMgRmlsZXMgYW5kIHVzZQoKLSBgY2FuZGlkYXRlLmlweW5iYDogY2xlYXJlZC1vdXRwdXQsIHNlbGYtY29udGFpbmVkIG5vdGVib29rIGVtYmVkZGluZyB0aGUgZXhhY3QgcmV2aWV3ZWQgc291cmNlIFpJUCBhbmQgcnVubmVyIHN1cHBvcnQuIENvbmZpZ3VyZSB0aGUgZXhpc3Rpbmcgb2ZmaWNpYWwgY2hlY2tvdXQsIHB1YmxpYy1vbmx5IG1vZGVsIGNhY2hlLCBvdXRwdXQgcm9vdCBhbmQgYmFja2VuZDsgZXhwbGljaXRseSBzZXQgYFJVTl9HUFUgPSBUcnVlYCBvbmx5IG9uIHRoZSBhdXRob3JpemVkIGNsb3VkIFQ0LgotIGBjYW5kaWRhdGUuemlwYDogd29ya2VyIHVwbG9hZCBhcnRpZmFjdCBjb250YWluaW5nIG9ubHkgYHNvbHV0aW9uL2Agc291cmNlLiBJdCBpcyBkZXRlcm1pbmlzdGljIGFuZCBoYXMgbm8gY3JlZGVudGlhbHMuCi0gYHNvdXJjZV9tYW5pZmVzdC5qc29uYDogZXhhY3QgcGVyLXNvdXJjZSBTSEEyNTYgaGFzaGVzIGVtaXR0ZWQgYnkgdGhlIHF1ZXVlJ3MgYGJ1bmRsZV9jYW5kaWRhdGUucHlgLgotIGBzdWJtaXNzaW9uLnRlbXBsYXRlLmpzb25gOiBjb21wYXRpYmxlIHF1ZXVlIHRlbXBsYXRlLiAqKktlZXAgdGhlIEdDUCBiYXNlbGluZSBJRCB1bnJlc29sdmVkIHVudGlsIGEgcmVhbCBzYW1lLWVudmlyb25tZW50IGJhc2VsaW5lIGlzIHJlZ2lzdGVyZWQuKiogTm8gcXVldWUgb3BlcmF0aW9uIHdhcyBwZXJmb3JtZWQuIFRoZSBjdXJyZW50IHdvcmtlciBzdXBwb3J0cyBgc291cF9zY3JlZW5fdjFgOyB0aGUgbm90ZWJvb2sgc2VwYXJhdGVseSBydW5zIHBhaXJlZCB2YWxpZGF0aW9uLCBub3QgYSBmaWN0aW9uYWwgcGFpcmVkIHF1ZXVlIHByb2ZpbGUuCi0gYHJ1bl9wcmVwYXJlZC5weWA6IG5vdGVib29rIHJ1bm5lci4gYGNoZWNrX2thZ2dsZV9lbnZpcm9ubWVudC5weWAgYW5kIGBwcmVwYXJlX2Fzc2V0X21hbmlmZXN0LnB5YCBhcmUgdW5tb2RpZmllZCBxdWV1ZSBzdXBwb3J0LCBub3QgY2xhaW1lZCBvcmlnaW5hbCBvcHRpbWl6YXRpb24ga2VybmVscy4KLSBgb2ZmaWNpYWxfZnJvemVuX21hbmlmZXN0Lmpzb25gOiBoYXNoZXMgb2YgbG9jYWwgb2ZmaWNpYWwgaGFybmVzcy90YXNrL3Rlc3RzIGF0IGNvbW1pdCBgYzdjMmZjZDE1OTcyY2U4Yjg5OGQ4ZDNmZjczMmVlMTkyYzgxN2NlYmAuCi0gYHZhbGlkYXRpb24uanNvbmAsIGBjcHVfaW50ZXJwcmV0ZXIubG9nYCwgYGNwdV9uYXRpdmUubG9nYCwgYG9mZmljaWFsX2NwdV9yZWdyZXNzaW9uLmxvZ2AsIGBzdGF0aWNfY2hlY2tzLmxvZ2AsIGBydW5uZXJfY2xlYW51cC5sb2dgOiBsb2NhbCBldmlkZW5jZSBvbmx5LgoKTm8gcGFja2FnZXMvbW9kZWwvZGF0YSBhcmUgaW5zdGFsbGVkIG9yIGRvd25sb2FkZWQuIFRoZSBvZmZsaW5lIHJ1bm5lciB2ZXJpZmllcyBleGFjdCBkaXJlY3QgcGlucywgZm91ci1DUFUgYWZmaW5pdHksIG9uZSB2aXNpYmxlIFRlc2xhIFQ0LCBvZmZpY2lhbCBjb21taXQvZnJvemVuIGZpbGUgaGFzaGVzLCBleGlzdGluZyBwdWJsaWMgZGF0YSBoYXNoZXMgYW5kIGNhY2hlZCBtb2RlbCByZXZpc2lvbi4gSXQgZXh0cmFjdHMgYSAqKm5ldyoqIGxvY2FsIGNvcHkgb2YgdGhlIG9mZmljaWFsIGNvbW1pdCBhbmQgb25seSByZXBsYWNlcyB0aGF0IGNvcHkncyBgc29sdXRpb24vYDsgaXQgbmV2ZXIgZWRpdHMgdGhlIHByZXBhcmVkIGNoZWNrb3V0LiBUaGUgb2ZmaWNpYWwgcGFpcmVkIGhhcm5lc3MgKGAtLXJlcGVhdHMgMmAsIHNlZWQgMSkgaXMgdW5jaGFuZ2VkIGFuZCBjcmVhdGVzIGZyZXNoIHRyYWluaW5nIHByb2Nlc3Nlcy9zdGF0ZS9vdXRwdXQvY29tcGlsZXIgY2FjaGVzLCB0aW1lcyB3aG9sZSB0cmFpbmluZyBjb21tYW5kcywgZXh0ZXJuYWxseSBzYW1wbGVzIG52aWRpYS1zbWksIHBlcmZvcm1zIHB1YmxpYyBxdWFsaXR5IGV2YWx1YXRpb24gYW5kIG1lYXN1cmVzIG93bmVkIGtlcm5lbHMgYWZ0ZXIgdHdvIHdhcm11cCBzdGVwcy4gUHJlZmxpZ2h0IHVzZXMgYSBzZXBhcmF0ZSBwcm9jZXNzIGFuZCBjb21waWxlciBjYWNoZS4gVGhlIGRlZmF1bHQgbmF0aXZlIHRpbWVvdXQgaXMgOTAwIHNlY29uZHMgZm9yIGVpdGhlciBiYWNrZW5kLCBub3QgdGhlIGhpc3RvcmljYWwgNzktc2Vjb25kIEthZ2dsZSBzY3JlZW4uIEEgR0NQIHJ1biBuZXZlciBpbmhlcml0cyBhIEthZ2dsZSBiYXNlbGluZSBvciBvZmZpY2lhbCBhZG1pc3Npb24uCgpUaGUgcnVubmVyIHRoZW4gaW52b2tlcyB1bmNoYW5nZWQgYGhhcm5lc3MvY2hlY2tfcnVsZXMucHlgLCBpbmNsdWRpbmcgb3JpZ2luYWwgcmVncmVzc2lvbiB0ZXN0cy4gRmFpbGVkLCBwYXJ0aWFsIG9yIHRpbWVkLW91dCBydW5zIHJldGFpbiBsb2dzIGFuZCBjYW5ub3QgcmVjZWl2ZSBhIHNjb3JlL2FkbWlzc2lvbiBQQVNTLiBSYXcgbG9ncyBtYXkgY29udGFpbiBvZmZpY2lhbCBoYXJuZXNzIHN1bW1hcmllcywgYnV0IGByZXZpZXdfc3VtbWFyeS5qc29uYCBpbmRlcGVuZGVudGx5IHJlamVjdHMgaW5jb21wbGV0ZS9mYWlsZWQgbmVzdGVkIHJlc3VsdHMgYW5kIGFsd2F5cyByZXRhaW5zIHNjb3JlIG51bGwvYWRtaXNzaW9uIFVOS05PV04uIENvbXBsZXRlIHBhaXJlZCBtZWFzdXJlbWVudHMgYXJlIHJlc2VhcmNoIGV2aWRlbmNlLCBub3QgdGhlIGZpbmFsIG9mZmljaWFsIEthZ2dsZSBqdWRnbWVudC4gVGhlIGNhbm9uaWNhbCBub25ibG9ja2luZyBgL3RtcC9zb3VwLWdwdTAubG9ja2Agc2VyaWFsaXplcyB0aGUgZW50aXJlIHBhaXJlZCBydW4gYWNyb3NzIGFsbCBzaXggY2FuZGlkYXRlIG5vdGVib29rcy4gU2hvcnQtbGl2ZWQgR1BVIHByZWZsaWdodCBleGl0cyBiZWZvcmUgYSBjb21wdXRlLXByb2Nlc3MgaWRsZSBjaGVjazsgdGhlIHBhcmVudCBydW5uZXIgbmV2ZXIgaW1wb3J0cyBUb3JjaCBvciBob2xkcyBhIENVREEgY29udGV4dC4gQSBMaW51eCBjaGlsZC1zdWJyZWFwZXIgdHJhY2tzIG93bmVkIGRlc2NlbmRhbnRzIGFjcm9zcyB0aGUgb2ZmaWNpYWwgaGFybmVzcydzIG5lc3RlZCBzZXNzaW9ucyBhbmQga2lsbHMvcmVhcHMgdGhlbSBvbiB0aW1lb3V0IG9yIGludGVycnVwdCBiZWZvcmUgcmVsZWFzaW5nIHRoZSBHUFUgbG9jay4gTm9ybWFsLWV4aXQgb3JwaGFuIGxlYWthZ2UgaXMgcmVqZWN0ZWQgdG9vLiBOZXN0ZWQtc2Vzc2lvbiB0aW1lb3V0LCBub3JtYWwtcGFyZW50LWV4aXQgb3JwaGFuIGFuZCBTSUdURVJNIGNsZWFudXAgd2VyZSBDUFUtdGVzdGVkIHVzaW5nIHBpZGZkLXNhZmUgc2lnbmFsaW5nLiBBbGwgbGFuZXMgY2hlY2sgYC90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZGAgYmVmb3JlIGFuZCBhZnRlciBhY3F1aXJpbmcgdGhlIGxvY2suIFRoaXMgcnVubmVyIHdyaXRlcyBhbiBvd25lci10b2tlbiBjcmFzaCBtYXJrZXIgYmVmb3JlIHN1YnByb2Nlc3Mgd29yazsgbm9ybWFsIGNsZWFuIHRlcm1pbmF0aW9uIGNhbiBjbGVhciBvbmx5IGl0cyBvd24gdG9rZW4uICoqT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24gb3IgdW5jZXJ0YWluIGNsZWFudXAgcmV0YWlucyBxdWFyYW50aW5lIGV2ZW4gYWZ0ZXIgb2JzZXJ2ZWQgcHJvY2VzcyBjbGVhbnVwOyBvcGVyYXRvciByZWNvbmNpbGlhdGlvbiBpcyByZXF1aXJlZCwgbmV2ZXIgYXV0b21hdGljIGNsZWFyaW5nLioqIE1hcmtlciBwZXJzaXN0ZW5jZS9yZWZ1c2FsIHdhcyBDUFUtdGVzdGVkIHdpdGggaXNvbGF0ZWQgZml4dHVyZSBwYXRocy4gSGFyZC1raWxsZWQgcnVubmVyL2hvc3QgZmFpbHVyZXMgbGlrZXdpc2UgcmVxdWlyZSBvcGVyYXRvciByZXNvdXJjZSBpbnNwZWN0aW9uIGJlZm9yZSBhbnkgcmV0cnkuIERvIG5vdCBydW4gbXVsdGlwbGUgR1BVIG5vdGVib29rcyBjb25jdXJyZW50bHkuCgojIyBJbXBsZW1lbnRhdGlvbiBjb250cmFjdAoKYHNvbHV0aW9uL3JvdGFyeS5weWAgb3ducyB0d28gbGF1bmNoZWQga2VybmVscywgYHJvdGFyeV9hcHBseV9mMzJgIGFuZCBgcm90YXJ5X2Fkam9pbnRfZjMyYCwgcGx1cyBhbiBpbmxpbmVkIHNpZ24tYml0IGhlbHBlci4gTm8gc3RhcnRlciBvciByZWFkeS1tYWRlIGtlcm5lbCBpcyBjb3BpZWQgb3IgY3JlZGl0ZWQuIFRoZSBzdGFydGVyIFN3aUdMVSBwYXRjaCBpcyBub3QgaW5zdGFsbGVkOyB0aGUgYmFzZWxpbmUgTUxQIHJlbWFpbnMgbmF0aXZlLgoKLSBDVURBIEZQMzIgUS9LL2NvZWZmaWNpZW50IGd1YXJkOyBwcm9kdWN0aW9uIHNoYXBlcyBCPD00LCBUPD01MTIsIEhxL0hrPTEyLzIsIEQ9MTI4OyBkZW5zZSB0cmFuc3Bvc2VkIEJUSEQgc3RvcmFnZSwgY29udGlndW91cyBjb2VmZmljaWVudHMgd2l0aCBCcG9zPTEgb3IgQjsgZGVmYXVsdCB1bnNxdWVlemUgYXhpcyBvbmx5LgotIEZyZXNoIG91dHB1dHMgd2l0aCBpbnB1dCBzdHJpZGVzLCB1bm1vZGlmaWVkIGlucHV0cywgZnJvemVuIHN1cHBsaWVkIGNvZWZmaWNpZW50cywgaW5kZXBlbmRlbnQgY29lZmZpY2llbnQgaGFsdmVzIGFuZCBRL0sgcmVxdWlyZXMtZ3JhZCBicmFuY2hlcy4gTm8gcmV0YWluZWQgUS9LIGFjdGl2YXRpb24gc3RvcmFnZSBjbGFpbTogb25seSBjb2VmZmljaWVudHMgYXJlIHNhdmVkLCBhbmQgdGhlIGJhc2VsaW5lIGFscmVhZHkgc2hhcmVzIHRoZWlyIHN0b3JhZ2UuCi0gU2hhcmVkIGFjdGl2YXRpb24gc3RvcmFnZSwgdGVuc29yIHN1YmNsYXNzZXMsIGxhenkgbmVnYXRpdmUvY29uanVnYXRlIGlucHV0IHZpZXdzLCB0cmFpbmFibGUgY29lZmZpY2llbnRzLCB1bnN1cHBvcnRlZCBsYXlvdXQvZHR5cGUvc2hhcGUvYXhpcywgYW5kIENQVSBjYWxscyB1c2UgdGhlIGNhcHR1cmVkIG5hdGl2ZSBmdW5jdGlvbi4gTW9kdWxlIGluc3RhbGxhdGlvbiBpcyBoYXNoLWd1YXJkZWQgYWdhaW5zdCB0aGUgcGlubmVkIFRyYW5zZm9ybWVycyA1LjE2LjEgUXdlbjIgc291cmNlIGJsb2IuIFVua25vd24gc291cmNlIGZhaWxzIGluc3RhbGxhdGlvbiwgbm90IHRyYWluaW5nIHVuZGVyIGFuIHVuaWRlbnRpZmllZCBwYXRjaC4KLSBTZXBhcmF0ZSBGUDMyIHByb2R1Y3RzIGFuZCBhZGRpdGlvbiAoYGVuYWJsZV9mcF9mdXNpb249RmFsc2VgKS4gVW5hcnkgbmVnYXRpb24gZmxpcHMgdGhlIEZQMzIgc2lnbiBiaXQsIG5vdCBgMC14YC4gQmFja3dhcmQgdXNlcyB0aGUgKipjcm9zcy1oYWxmIHNpbmUqKiBjb2VmZmljaWVudCBhbmQgbmF0aXZlIHNsaWNlLXBhZGRpbmcgemVybyBjYW5vbmljYWxpemF0aW9uIGBaKHopPSswIGlmIHo9PTAgZWxzZSB6YC4gVGhpcyBwcmVzZXJ2ZXMgdGhlIGF1ZGl0ZWQgbmF0aXZlIGFkam9pbnQgZ3JhcGggcmF0aGVyIHRoYW4gc3Vic3RpdHV0aW5nIGEgZ2VuZXJpYyBpbnZlcnNlIHJvdGF0aW9uLgotIEJhY2t3YXJkIGFjY2VwdHMgYXJiaXRyYXJ5IHVwc3RyZWFtIHN0cmlkZXMgYW5kIHJlc29sdmVzIGxhenkgbmVnYXRpdmUvY29uanVnYXRlIHZpZXdzLiBVbnVzZWQgb3V0cHV0cyBrZWVwIE5vbmUgZ3JhZGllbnRzLiBDb2VmZmljaWVudHMgdXNlIGBzYXZlX2Zvcl9iYWNrd2FyZGAgZm9yIHZlcnNpb24gY2hlY2tzLiBIaWdoZXItb3JkZXIgcmVxdWVzdHMgdXNlIGEgZGlmZmVyZW50aWFibGUgbmF0aXZlIFB5VG9yY2ggYWRqb2ludCBpbnN0ZWFkIG9mIGFuIG9wYXF1ZSBUcml0b24gc2Vjb25kIGRlcml2YXRpdmUuCi0gRmlyc3QgYW5kIGV2ZXJ5IDI1NnRoIHJvdXRlL2xhdW5jaCBldmVudCBwcmludCBmbHVzaGVkIGBST1RBUllfRVZJREVOQ0VgIGxpbmVzLiBUaGV5IHN1cnZpdmUgbmF0aXZlIHRpbWVvdXRzIGluIHRoZSBvZmZpY2lhbCBgdHJhaW4ubG9nYC4gQSBgbGF1bmNoX3JldHVybmVkYCBldmVudCBpcyBhIGhvc3QgbGF1bmNoIG9ic2VydmF0aW9uLCAqKm5vdCBwcm9vZiBvZiBjb21wbGV0ZWQgQ1VEQSBleGVjdXRpb24qKjsgY29tcGxldGVkIG9mZmljaWFsIG1ldHJpY3MvcXVhbGl0eSBjaGVja3MgYXJlIHN0aWxsIHJlcXVpcmVkLgoKVGhlIEdQVSBwcmVmbGlnaHQgcmVxdWlyZXMgZXhhY3QgYXJpdGhtZXRpYyAoaW5jbHVkaW5nIHplcm8gc2lnbnMpLCBleGhhdXN0aXZlIHNpZ25lZC16ZXJvIGJvdW5kYXJ5IGNhc2VzLCBub25maW5pdGUgY2xhc3Nlcy9zdWJub3JtYWxzLCByZWFsaXN0aWMgZnVsbCBzaGFwZXMsIHVuY2hhbmdlZCBtYXNrZWQvdW5tYXNrZWQgU0RQQSBhdXRvY2FzdC9HUUEsIHJlYWwgUXdlbjJBdHRlbnRpb24gcm91dGUsIGFuZCBQVFggd2l0aCBzZXBhcmF0ZSBGUDMyIG11bHRpcGx5L2FkZCwgbm8gRk1BIGFuZCBubyBGVFouIEl0IGV4cG9ydHMgUFRYIGZvciBodW1hbiByZXZpZXcuIEdQVSB0ZXN0cyBhcmUgc3VwcGxpZWQgYnV0ICoqbmV2ZXIgcnVuIGhlcmUqKi4gTmFOIHBheWxvYWQtYml0IGlkZW50aXR5IGFuZCBmdWxsLW1vZGVsIGJpdHdpc2UgaWRlbnRpdHkgYXJlIG5vdCBwcm9taXNlZC4gU2hhcmVkLWFuY2VzdHJ5IGFjY3VtdWxhdGlvbiBjYW4gZGlmZmVyIGV2ZW4gd2hlbiBzdGFuZGFsb25lIFJvUEUgaXMgZXhhY3Q7IGludGVncmF0aW9uIHRlc3RzIGNvbXBhcmUgcmF3IHNjYWxlZCBncmFkaWVudHMgd2l0aCBkZWNsYXJlZCB0b2xlcmFuY2VzLCBub3QgdW5zY2FsZWQgZ3JhZGllbnRzLgoKIyMgU2NpZW50aWZpYyBsaW1pdHMgYW5kIHJlamVjdGlvbiBjcml0ZXJpYQoKVGhlIHN0cm9uZ2VzdCBvcGVuIGJsb2NrZXIgaXMgZWxpZ2liaWxpdHk6IHRoaXMgbmFycm93IG9wZXJhdGlvbiBtYXkgYmUgdG9vIHNtYWxsIHRvIHByb3ZpZGUgdGhlIHJlcXVpcmVkICoqPj01JSB1c2VmdWwgb3JpZ2luYWwta2VybmVsIHdhbGwgc2hhcmUqKi4gTm8gYXJ0aWZpY2lhbCB3b3JrIGlzIGFkZGVkIHRvIHJlYWNoIHRoYXQgdGhyZXNob2xkLiBSZWplY3QgaWYgcm91dGluZyBpcyBhYnNlbnQsIHVzZWZ1bCBzaGFyZSBpcyBpbnN1ZmZpY2llbnQsIGZhbGxiYWNrIGRvbWluYXRlcywgUFRYL251bWVyaWNzIGZhaWwsIGZ1bGwtdGFzayBxdWFsaXR5L291dHB1dHMgZGl2ZXJnZSwgb3IgY29tcGxldGUgdGltZS9tZW1vcnkgaXMgd29yc2UgYmV5b25kIG9mZmljaWFsIGxpbWl0cy4gVGhlcmUgaXMgbm8gR1BVLW9wdGltYWxpdHksIHNwZWVkdXAsIHJldGFpbmVkLVEvSyBzYXZpbmdzIG9yIG1lYXN1cmVkIFBhcmV0byBjbGFpbS4KClJlc2VhcmNoIGlucHV0cyBpbnNwZWN0ZWQgbG9jYWxseTogYHNvdXBfY3ljbGUvcmVzZWFyY2hfcm9wZV9jeWNsZTMubWRgLCBpdHMgbmF0aXZlIHJvdXRpbmcvQ1BVIGNvbnRyYWN0czsgYHNvdXBfYXVkaXRfMTkxOS9xd2VuLmpzb25gOyBgc291cF9hdWRpdF8xOTE5L2hhbmRvZmYuanNvbmAgKEhBTkRPRkYtUk9QRS1TSUdORURaRVJPKTsgYW5kIHN1cHBsaWVkIGBzb3VwX3VzZXJfZmluZGluZ3NfMjAyNjEwMzAvdGVzdF9yb3RhcnkucHlgLiBUaGV5IGluZm9ybSByZXF1aXJlbWVudHMsIG5vdCBydW50aW1lIG93bmVyc2hpcCBvciBpbmhlcml0ZWQgdmFsaWRhdGlvbi4gQWxsIHRlc3RzIGluIHRoaXMgcGFja2V0IGFyZSBpbmRlcGVuZGVudGx5IGF1dGhvcmVkLCBhbmQgZWFybGllciBzdXBwbGllZCB0ZXN0cyB3ZXJlIG5vdCByZWxhYmVsZWQgYXMgdGhpcyBpbXBsZW1lbnRhdGlvbidzIHJlc3VsdHMuCgojIyBSZXByb2R1Y2UgbG9jYWwgQ1BVIGNoZWNrcyAobm8gR1BVKQoKYGBgYmFzaApDVURBX1ZJU0lCTEVfREVWSUNFUz0nJyBPTVBfTlVNX1RIUkVBRFM9MiBUUklUT05fSU5URVJQUkVUPTEgXAogIC9yZWRhY3RlZC9sb2NhbC9zb3VwX2Nsb3VkX2Vudi92ZW52L2Jpbi9weXRob24gLW0gcHl0ZXN0IHNvbHV0aW9uL3Rlc3Rfcm90YXJ5LnB5IC1xCkNVREFfVklTSUJMRV9ERVZJQ0VTPScnIC9yZWRhY3RlZC9sb2NhbC9zb3VwX2Nsb3VkX2Vudi92ZW52L2Jpbi9weXRob24gLW0gc29sdXRpb24uY3B1X2NvbnRyYWN0CnB5dGhvbiB2YWxpZGF0ZV9hcnRpZmFjdHMucHkKYGBgCgpUaGUgYXZhaWxhYmxlIGludGVycHJldGVyIGVtaXRzIG9uZSBleHBlY3RlZCBOdW1QeSBpbnZhbGlkLW11bHRpcGx5IHdhcm5pbmcgb24gZGVsaWJlcmF0ZSBpbmZpbml0aWVzL05hTnMuIEludGVycHJldGVyIGV4ZWN1dGlvbiBpcyBub3QgY29tcGlsYXRpb24sIHRhcmdldC12ZXJzaW9uIHBhcml0eSBvciBDVURBIGV2aWRlbmNlLiBEbyBub3QgaW5zdGFsbC9kb3duZ3JhZGUgcGFja2FnZXMgdG8gZm9yY2UgdGhvc2UgY2hlY2tzIGludG8gYSB0YXJnZXQgY2xhaW0uCg=='}
PAYLOAD_SHA256 = {'candidate.zip': 'b874e6a4f23b9912014ed3f39e30c44716fc3af5aece742d5c03d447643d3829', 'source_manifest.json': 'f86cd383d1ccb49f4ef473cd672b3be9609f5007e7e7be07d2a477b396106672', 'official_frozen_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'run_prepared.py': '3ad54e3ae1eb9abc6edc429752b35698e1e5ecaf2f8a177f72734fd805056753', 'check_kaggle_environment.py': '0d3d40996a1823f22f0f487649cb39e72357262cf7b39dcb6ae4200acdf7d5a3', 'prepare_asset_manifest.py': 'f2a0f36b5cd367d506c831c69f1a39d49c38c3c815758be5d0fd7a2ec033f46d', 'requirements-kaggle.txt': '1159a601742f7d6e745b5e878a12e12c465dba414690e98bc70c4cb02ebbac2d', 'submission.template.json': '2716d36362702cf6fc706140f89dfa13741880a92a0989fef0227e6333fb119c', 'README.md': '59794f6d6032766e43334a966db2a51637432f4e6dc4773e6a573cadedc1c78d'}
STAGED = Path(tempfile.mkdtemp(prefix="rotary-reviewed-candidate-"))
for name, encoded in PAYLOAD.items():
    data = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(data).hexdigest() == PAYLOAD_SHA256[name], name
    (STAGED / name).write_bytes(data)
manifest = json.loads((STAGED / "source_manifest.json").read_text())
with zipfile.ZipFile(STAGED / "candidate.zip") as archive:
    assert set(archive.namelist()) == set(manifest)
    for name, wanted in manifest.items():
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == wanted, name
        if name.endswith(".py"):
            compile(data, name, "exec")
print("Reviewed bundle:", PAYLOAD_SHA256["candidate.zip"])
print("Staged safely at:", STAGED)
print("GPU status: NOT RUN; score: null; admission: UNKNOWN")


## Future GPU preflight and unchanged paired official task
The runner requires exact direct package pins/Python, official commit, fixed public data and retained model revision. It creates a fresh local official tree, replaces only `solution/`, and holds the canonical GPU0 lock. GPU preflight is a separate process/cache; it must pass arithmetic, signed-zero/native-adjoint, unchanged SDPA/GQA routing and PTX no-contraction/FTZ checks before training.

Then the unchanged official harness runs base/solution/solution/base (two repeats, fixed seed 1) with fresh training state/output/compiler caches, whole-command wall time, external nvidia-smi peak, public evaluation and native kernel-share telemetry after two warmups. Original regression checks run unchanged. Timeout/partial/failure logs are retained; no score or automatic official admission is assigned. `ROTARY_EVIDENCE` launch-returned lines survive timeouts but are not CUDA-completion proof.

This notebook has never been executed on a GPU. Default execution stops at the explicit opt-in guard below.


In [ ]:
import subprocess, sys, signal
if not RUN_GPU:
    raise RuntimeError("Prepared only. Review README/source, configure existing cloud paths, then explicitly opt in with RUN_GPU=True.")
command = [sys.executable, str(STAGED / "run_prepared.py"),
           "--prepared-repo", str(PREPARED_REPO), "--hub-cache", str(PUBLIC_HUB_CACHE),
           "--run-root", str(RUN_ROOT), "--backend", BACKEND,
           "--timeout", str(NATIVE_TIMEOUT_SECONDS), "--confirm-cloud-gpu-run"]


runner_process = subprocess.Popen(command, start_new_session=True)
try:
    return_code = runner_process.wait()
except BaseException:
    if runner_process.poll() is None:
        runner_process.send_signal(signal.SIGTERM)

    print("Stopping the runner and waiting for its owned child cleanup...")
    runner_process.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
